### CSCI-E104: Advanced Deep Learning, Spring 2026
### Lab 7: Text-to-Speech (TTS) 
**Harvard University Extension School - Prof. Zoran B. Djordjević, Blagoje Djordjević**<br/>
**Joan Imrich  13-March-2026**<br/>
<hr style="height:2pt">

## <a class="voice-clone-demo" id="voice-clone-demo"> ((((((((((((( TTS AAC Speech Assistant ))))))))))) </a> 

## <font color="#DC143C"> TTS - Augmentative and Alternative Communication (AAC)</font>
**This demo may be helpful for  <font color="#DC143C"> Final Project Ideas?</font>**

<font color="#DC143C"> **AAC Speech Assistant** </font> is an interactive Jupyter notebook app that helps someone speak common phrases by clicking categorized buttons, which then generate synthetic speech audio using a **multilingual XTTSv2 text‑to‑speech model**

## AAC Speech Assistant
- Notebook builds a small **speech-generating** interface for **Augmentative and Alternative Communication (AAC).** 
- AAC tools support people who have difficulty producing clear speech by giving them alternative ways to express needs, feelings, and social phrases. [See asha.org details](https://www.asha.org/practice-portal/professional-issues/augmentative-and-alternative-communication/)

- Phrases are grouped into categories such as greetings, basic needs, communication, emotions, and actions (e.g., “Hello”, “Water”, “Help”, “Happy”, “Go”).  
- Each phrase becomes a button in an ipywidgets tabbed interface (Tabs for “Greetings”, “Needs”, “Communication”, etc.), so the user can quickly find what they want to say.  
- When a button is pressed, the phrase text is sent to a TTS engine, audio is generated into a temporary WAV file, loaded as bytes, and played via an `Audio` widget.  
- There are control buttons to clear the output, repeat the last phrase, and list all available phrases.  
- The whole layout (`main_display`) is shown as a vertical box: title, instructions, tabs, control buttons, audio player, and a text output area for status messages.


## XTTS model overview, code uses XTTSv2:
- Conceptually, this is a high‑tech AAC device: instead of a physical communication board, you have a software board, and instead of digitized recorded clips, you use synthetic speech generated on demand. 
- All of this complexity is wrapped behind the `tts.tts_to_file(...)` call: you give it text, a speaker, and a language, and it returns an audio file that the AAC interface can play back.

```python
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=False)
```

**XTTS (as implemented in Coqui TTS)** is a **multilingual, multi-speaker text‑to‑speech model** designed to produce natural‑sounding speech in several languages from arbitrary text. It supports “voice cloning” or voice transfer: given a reference speaker, the model can approximate that speaker’s voice while speaking new text in supported languages. In your notebook you pass a speaker name (e.g., `"Ana Florence"`) and language `"en"` to generate English speech audio from AAC phrases.

Compared with older concatenative or simple parametric systems, XTTS uses a modern neural architecture that can model prosody (intonation, rhythm) and accent more naturally, which makes AAC output clearer and less robotic, and potentially more comfortable for everyday communication. 

## XTTS-style architecture (high level)
While exact internals depend on the specific XTTS version, architectures in this family generally have three main parts:

1. **Text encoder**  
   - Converts input text into a sequence of linguistic embeddings (tokens with phonetic and punctuation information).  
   - Captures context so the model can decide prosody (where to pause, how to emphasize words).

2. **Speaker / language conditioning**  
   - A speaker encoder or learned speaker embedding captures voice characteristics (timbre, pitch range, speaking style).  
   - A language embedding indicates which language to speak, allowing one model to handle multiple languages.  
   - These conditioning vectors are fused with the text representation so the same words can be spoken in different voices and languages.

3. **Acoustic and vocoder stages**  
   - An acoustic model (often based on transformers or similar sequence models) predicts intermediate acoustic features such as mel-spectrograms from the conditioned text sequence.  
   - A neural vocoder (e.g., a GAN- or diffusion-based vocoder) then converts these spectrograms into raw audio waveforms in real time or near real time.


In [1]:
# Verify environment
import sys
import os
import torch

print("Conda env:", os.environ.get('CONDA_DEFAULT_ENV', 'NONE'))
print("Python path:", sys.executable)
print("Python version:", sys.version)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

import TTS
print("TTS location:", TTS.__file__)  # MUST be in /envs/coquitts
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

print("Python:", sys.version.split()[0])
print("coquitts:", "coquitts" in sys.executable)
print("numpy:", __import__('numpy').__version__)

Conda env: xtts-vc
Python path: /home/csci/anaconda3/envs/xtts-vc/bin/python
Python version: 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:45:40) [GCC 14.3.0]
PyTorch CUDA: 12.1
CUDA available: True
TTS location: /home/csci/anaconda3/envs/xtts-vc/lib/python3.11/site-packages/TTS/__init__.py
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
Python: 3.11.15
coquitts: False
numpy: 1.26.4


In [2]:
# Cell 1: Setup (run once)

import ipywidgets as widgets
from IPython.display import display, Audio, clear_output, HTML
import tempfile
import os
from TTS.api import TTS

# Initialize TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=False)

# AAC Phrases
AAC_PHRASES = {
    "Greetings": {
        "Hello": "Hello, how are you today?",
        "Goodbye": "Goodbye, see you later!",
        "Thank you": "Thank you very much!"
    },
    "Needs": {
        "Water": "I want water please.",
        "Food": "I'm hungry. Food please.",
        "Bathroom": "I need the bathroom.",
        "Sleep": "I want to sleep."
    },
    "Communication": {
        "Yes": "Yes, that's right.",
        "No": "No, that's not right.",
        "Help": "I need help please.",
        "Stop": "Please stop that."
    },
    "Emotions": {
        "Happy": "I am happy!",
        "Sad": "I am sad.",
        "Tired": "I am tired."
    },
    "Actions": {
        "Go": "Let's go!",
        "Wait": "Please wait.",
        "More": "I want more please."
    }
}

print("AAC TTS Assistant Ready!")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
AAC TTS Assistant Ready!


In [3]:
# Cell 2: FIXED - Complete UI with WORKING "All Phrases" button
current_audio = b''
current_phrase = ""
output = widgets.Output()

def speak_phrase(phrase):
    global current_audio, current_phrase
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            temp_path = tmp.name
        tts.tts_to_file(text=phrase, file_path=temp_path, speaker="Ana Florence", language="en")
        with open(temp_path, 'rb') as f:
            current_audio = f.read()
        os.unlink(temp_path)
        current_phrase = phrase
        return current_audio
    except Exception as e:
        print(f"TTS Error: {e}")
        return None

# Create tabs
tab_contents = []
tab_names = list(AAC_PHRASES.keys())

for category_name, phrases in AAC_PHRASES.items():
    buttons = []
    for label, phrase in phrases.items():
        btn = widgets.Button(
            description=label,
            layout=widgets.Layout(width='140px', height='60px', font_size='14px'),
            style={'button_color': '#4CAF50'}
        )
        
        def make_click_handler(phrase=phrase, label=label, category=category_name):
            def on_click(b):
                b.style.button_color = '#FF9800'
                audio = speak_phrase(phrase)
                if audio:
                    with output:
                        clear_output()
                        display(HTML(f"""

                        <div style="background: #87CEFA); 
                                   color: black; padding: 20px; border-radius: 15px; 
                                   text-align: center; font-size: 24px; font-weight: bold;">
                            <div style="font-size: 20px;">"{phrase}"</div>
                        </div>
                        """))
                        print(f" Category: {category}")
                audio_widget.value = audio or b''
                b.style.button_color = '#4CAF50'
            return on_click
        
        btn.on_click(make_click_handler())
        buttons.append(btn)
    
    grid = widgets.GridBox(
        buttons, 
        layout=widgets.Layout(grid_template_columns="repeat(3, 140px)", grid_gap="10px")
    )
    tab_contents.append(grid)

# Create tabs
tabs = widgets.Tab()
for i, name in enumerate(tab_names):
    tabs.children += (tab_contents[i],)
    tabs.set_title(i, name)

# FIXED CONTROL BUTTONS - All connected properly
def on_clear(b): 
    global current_audio
    current_audio = b''
    audio_widget.value = b''
    output.clear_output()

def on_repeat(b): 
    global current_audio, current_phrase
    if current_audio:
        audio_widget.value = current_audio
        with output:
            clear_output()
            display(HTML(f'<div style="font-size:24px; color:#4CAF50">Repeating: "{current_phrase}"</div>'))

def on_list(b):  # THIS WAS MISSING THE CALLBACK CONNECTION
    """Show all phrases in a scrollable list"""
    with output:
        clear_output()
        display(HTML("""
        <div style="background: #f0f8ff; padding: 20px; border-radius: 10px; max-height: 400px; overflow-y: auto;">
            <h3 style="color: #2c3e50; margin-top: 0;">All Available Phrases</h3>
        """))
        
        for category, phrases in AAC_PHRASES.items():
            display(HTML(f'<h4 style="color: #34495e; border-bottom: 2px solid #3498db;">{category}</h4>'))
            for label, phrase in phrases.items():
                display(HTML(f'<div style="margin: 8px 0; padding: 10px; background: white; border-radius: 5px; border-left: 4px solid #3498db;">'
                           f'<strong>{label}:</strong> "{phrase}"</div>'))
        
        display(HTML("</div>"))

# CONNECT ALL BUTTONS
clear_btn = widgets.Button(description="Clear", button_style='warning', layout=widgets.Layout(width='120px'))
repeat_btn = widgets.Button(description="Repeat", button_style='info', layout=widgets.Layout(width='120px'))
list_btn = widgets.Button(description="List All Phrases", button_style='primary', layout=widgets.Layout(width='140px'))

clear_btn.on_click(on_clear)      # Connected
repeat_btn.on_click(on_repeat)    # Connected  
list_btn.on_click(on_list)        # NOW CONNECTED - This was missing!

audio_widget = widgets.Audio(value=b'', format='wav', autoplay=False, controls=True)



In [4]:
# Final display #display(main_ui)
# Cell 10: actually show the UI

#from IPython.display import display

main_display = widgets.VBox([
    widgets.HTML("<h1>AAC Speech Assistant</h1>"),
    widgets.HTML("<p style='font-size:18px; color:#666;'>Click buttons → See text + hear speech!</p>"),
    tabs,
    widgets.HTML("<hr style='border: 2px solid #4CAF50;'>"),
    widgets.HBox([clear_btn, repeat_btn, list_btn]),
    widgets.HTML("<h3>Audio Player:</h3>"),
    audio_widget,
    widgets.HTML("<h3>Spoken Text:</h3>"),
    output
], layout=widgets.Layout(width='100%', align_items='center'))


In [5]:
from IPython.display import display

display(main_display)

In [6]:
#! jupyter nbconvert --to html --execute TTS-aac.ipynb

In [7]:
# from ipywidgets.embed import embed_minimal_html

# # main_display is your VBox with tabs/buttons/audio_widget/...
# embed_minimal_html(
#     'aac_speech_assistant.html',
#     views=[main_display],
#     title='AAC Speech Assistant'
# )